In [2]:
import numpy as np
import pandas as pd
from collections import defaultdict

# ══════════════════════════════════════════════════════
#  NAIVE BAYES CLASSIFIER — HÀM TÁI SỬ DỤNG
# ══════════════════════════════════════════════════════

class NaiveBayesDA:
    """
    Naive Bayes cho Categorical Data + Laplace Smoothing.
    Tái sử dụng cho MỌI bài toán phân loại categorical.
    """

    def fit(self, X, y, feature_names):
        self.classes       = list(set(y))
        self.feature_names = feature_names
        n                  = len(y)

        self.prior = {
            c: sum(1 for yi in y if yi == c) / n
            for c in self.classes
        }

        self.cond = defaultdict(lambda: defaultdict(dict))
        for fi, fname in enumerate(feature_names):
            vals = list(set(X[:, fi]))
            for cls in self.classes:
                cls_count = sum(1 for yi in y if yi == cls)
                for v in vals:
                    cnt = sum(
                        1 for xi, yi in zip(X[:, fi], y)
                        if yi == cls and xi == v
                    )
                    self.cond[fname][v][cls] = (
                        (cnt + 1) / (cls_count + len(vals))
                    )
        return self

    def predict(self, x_input, verbose=True):
        scores = {}

        if verbose:
            print(f"\nInput: {dict(zip(self.feature_names, x_input))}")
            print("─" * 55)

        for cls in self.classes:
            score = self.prior[cls]
            if verbose:
                print(f"\n  Class='{cls}' | Prior={self.prior[cls]:.3f}")

            for fname, val in zip(self.feature_names, x_input):
                p = self.cond[fname].get(val, {}).get(cls, 0.001)
                score *= p
                if verbose:
                    print(f"    × P({fname}={val}|{cls}) = {p:.3f}")

            scores[cls] = score
            if verbose:
                print(f"    → Score = {score:.6f}")

        total = sum(scores.values())
        probs = {c: s / total for c, s in scores.items()}
        best  = max(probs, key=probs.get)

        if verbose:
            print(f"\n{'─'*55}")
            print("  POSTERIOR:")
            for c, p in sorted(probs.items(), key=lambda x: -x[1]):
                bar = "█" * int(p * 35)
                print(f"  {c}: {p:.1%}  {bar}")

        return best, probs


# ══════════════════════════════════════════════════════
#  DATASET 1: TENNIS (Đã train ở phần giải tay)
# ══════════════════════════════════════════════════════

tennis_data = [
    ['Sunny',    'Hot',  'High',   'Weak',   'No'],
    ['Sunny',    'Hot',  'High',   'Strong', 'No'],
    ['Overcast', 'Hot',  'High',   'Weak',   'Yes'],
    ['Rainy',    'Mild', 'High',   'Weak',   'Yes'],
    ['Rainy',    'Cool', 'Normal', 'Weak',   'Yes'],
    ['Rainy',    'Cool', 'Normal', 'Strong', 'No'],
    ['Overcast', 'Cool', 'Normal', 'Strong', 'Yes'],
    ['Sunny',    'Mild', 'High',   'Weak',   'No'],
    ['Sunny',    'Cool', 'Normal', 'Weak',   'Yes'],
    ['Rainy',    'Mild', 'Normal', 'Weak',   'Yes'],
]
TENNIS_FEATURES = ['Outlook', 'Temp', 'Humidity', 'Wind']
arr_tennis = np.array(tennis_data)

model_tennis = NaiveBayesDA().fit(
    arr_tennis[:, :-1], arr_tennis[:, -1], TENNIS_FEATURES
)

print("=" * 60)
print("  DATASET 1: TENNIS — Xác nhận kết quả giải tay")
print("=" * 60)
pred_tennis, probs_tennis = model_tennis.predict(
    ['Sunny', 'Cool', 'High', 'Strong']
)
print(f"\n  → Kết luận: {pred_tennis} "
      f"(P={probs_tennis[pred_tennis]:.1%})")


# ══════════════════════════════════════════════════════
#  DATASET 2: PHÊ DUYỆT VAY VỐN (Loan Approval)
#  Chứng minh hàm NaiveBayesDA() DÙNG LẠI được 100%
#  Chỉ cần đổi: data, features, input cần predict
# ══════════════════════════════════════════════════════

print("\n\n" + "=" * 60)
print("  DATASET 2: PHÊ DUYỆT VAY VỐN NGÂN HÀNG")
print("  (Thử hàm tái sử dụng với bài toán hoàn toàn khác)")
print("=" * 60)

loan_data = [
    # [Income,  CreditHistory, HasAsset, Approved]
    ['High', 'Good', 'Yes', 'Approved'],
    ['Low',  'Bad',  'No',  'Rejected'],
    ['High', 'Bad',  'No',  'Rejected'],
    ['Mid',  'Good', 'No',  'Approved'],
    ['Mid',  'Good', 'Yes', 'Approved'],
    ['Low',  'Good', 'No',  'Rejected'],
    ['High', 'Good', 'No',  'Approved'],
    ['Low',  'Bad',  'Yes', 'Rejected'],
    ['Mid',  'Bad',  'No',  'Rejected'],
    ['High', 'Good', 'Yes', 'Approved'],
    ['Low',  'Good', 'Yes', 'Approved'],
    ['Mid',  'Bad',  'Yes', 'Rejected'],
]
LOAN_FEATURES = ['Income', 'CreditHistory', 'HasAsset']
arr_loan = np.array(loan_data)

print("\n  DATASET:")
df_loan_display = pd.DataFrame(
    loan_data, columns=LOAN_FEATURES + ['Approved']
)
print(df_loan_display.to_string(index=False))

model_loan = NaiveBayesDA().fit(
    arr_loan[:, :-1], arr_loan[:, -1], LOAN_FEATURES
)

print("\n" + "─" * 60)
print("  TEST CASE: Khách hàng thu nhập thấp,")
print("  lịch sử tín dụng tốt, có tài sản thế chấp")
print("─" * 60)

new_customer = ['Low', 'Good', 'Yes']
pred_loan, probs_loan = model_loan.predict(new_customer)

print(f"\n  📋 KẾT LUẬN CHO VAY:")
print(f"  Quyết định: {pred_loan}")
print(f"  Độ tin cậy: {probs_loan[pred_loan]:.1%}")

if pred_loan == 'Approved':
    print(f"  ✅ NÊN phê duyệt khoản vay")
else:
    print(f"  ❌ KHÔNG nên phê duyệt khoản vay")


# ── TEST THÊM NHIỀU TRƯỜNG HỢP (Batch testing) ───────
print("\n" + "─" * 60)
print("  TEST NHIỀU KHÁCH HÀNG CÙNG LÚC")
print("─" * 60)

test_customers = [
    (['High', 'Good', 'Yes'], "KH A: Thu nhập cao, tốt mọi mặt"),
    (['Low',  'Bad',  'No'],  "KH B: Thu nhập thấp, xấu mọi mặt"),
    (['Mid',  'Good', 'No'],  "KH C: Trung bình, không tài sản"),
    (['High', 'Bad',  'Yes'], "KH D: Thu nhập cao nhưng lịch sử xấu"),
]

print(f"\n  {'Mô tả':<40} {'Dự đoán':>12} {'P(Approved)':>12}")
print("  " + "─" * 66)
for customer, desc in test_customers:
    pred, probs = model_loan.predict(customer, verbose=False)
    p_approved = probs.get('Approved', 0)
    icon = "✅" if pred == "Approved" else "❌"
    print(f"  {desc:<40} {icon} {pred:<9} {p_approved:>11.1%}")

print(f"\n  💡 INSIGHT:")
print(f"  → Hàm NaiveBayesDA() không cần sửa code")
print(f"  → Chỉ thay data + feature_names + input")
print(f"  → Dùng được cho: Tennis, Loan, Spam, Churn, ...")

  DATASET 1: TENNIS — Xác nhận kết quả giải tay

Input: {'Outlook': 'Sunny', 'Temp': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
───────────────────────────────────────────────────────

  Class='Yes' | Prior=0.600
    × P(Outlook=Sunny|Yes) = 0.222
    × P(Temp=Cool|Yes) = 0.444
    × P(Humidity=High|Yes) = 0.375
    × P(Wind=Strong|Yes) = 0.250
    → Score = 0.005556

  Class='No' | Prior=0.400
    × P(Outlook=Sunny|No) = 0.571
    × P(Temp=Cool|No) = 0.286
    × P(Humidity=High|No) = 0.667
    × P(Wind=Strong|No) = 0.500
    → Score = 0.021769

───────────────────────────────────────────────────────
  POSTERIOR:
  No: 79.7%  ███████████████████████████
  Yes: 20.3%  ███████

  → Kết luận: No (P=79.7%)


  DATASET 2: PHÊ DUYỆT VAY VỐN NGÂN HÀNG
  (Thử hàm tái sử dụng với bài toán hoàn toàn khác)

  DATASET:
Income CreditHistory HasAsset Approved
  High          Good      Yes Approved
   Low           Bad       No Rejected
  High           Bad       No Rejected
   Mid          Good  

In [3]:
# ============================================================
# CACH DUNG THONG NHAT: dataset nao cung theo cung mot mau
# ============================================================

def run_naive_bayes(name, data, feature_names, test_input):
    data_array = np.array(data)

    if data_array.ndim != 2 or data_array.shape[1] != len(feature_names) + 1:
        raise ValueError("data phai gom cac cot feature va 1 cot nhan o cuoi")
    if len(test_input) != len(feature_names):
        raise ValueError("test_input phai co dung so feature")

    model = NaiveBayesDA().fit(
        data_array[:, :-1], data_array[:, -1], feature_names
    )
    prediction, probabilities = model.predict(test_input, verbose=False)

    print(f"\n{name}")
    print(f"Input: {dict(zip(feature_names, test_input))}")
    print(f"Du doan: {prediction}")
    print(f"Do tin cay: {probabilities[prediction]:.1%}")
    return model, prediction, probabilities

model_tennis, pred_tennis, probs_tennis = run_naive_bayes(
    "TENNIS",
    tennis_data,
    TENNIS_FEATURES,
    ['Sunny', 'Cool', 'High', 'Strong'],
)

model_loan, pred_loan, probs_loan = run_naive_bayes(
    "LOAN APPROVAL",
    loan_data,
    LOAN_FEATURES,
    ['Low', 'Good', 'Yes'],
)


TENNIS
Input: {'Outlook': 'Sunny', 'Temp': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Du doan: No
Do tin cay: 79.7%

LOAN APPROVAL
Input: {'Income': 'Low', 'CreditHistory': 'Good', 'HasAsset': 'Yes'}
Du doan: Approved
Do tin cay: 74.5%


In [5]:
# ============================================================
# DATASET 3: DU BAO KHACH HANG ROI BO DICH VU (5 feature)
# ============================================================

churn_data = [
    # [Plan, Usage, Tickets, Contract, AutoPay, Churn]
    ['Basic',   'Low',  'Many', 'Monthly', 'No',  'Yes'],
    ['Basic',   'Low',  'Few',  'Monthly', 'No',  'Yes'],
    ['Basic',   'High', 'Few',  'Monthly', 'Yes', 'No'],
    ['Basic',   'High', 'Few',  'Annual',  'Yes', 'No'],
    ['Premium', 'High', 'Few',  'Annual',  'Yes', 'No'],
    ['Premium', 'High', 'Few',  'Monthly', 'Yes', 'No'],
    ['Premium', 'Low',  'Many', 'Monthly', 'No',  'Yes'],
    ['Premium', 'Low',  'Few',  'Annual',  'Yes', 'No'],
    ['Basic',   'Low',  'Many', 'Annual',  'No',  'Yes'],
    ['Premium', 'High', 'Many', 'Monthly', 'No',  'Yes'],
    ['Basic',   'High', 'Many', 'Monthly', 'Yes', 'Yes'],
    ['Premium', 'Low',  'Few',  'Monthly', 'Yes', 'No'],
]
CHURN_FEATURES = ['Plan', 'Usage', 'Tickets', 'Contract', 'AutoPay']

model_churn, pred_churn, probs_churn = run_naive_bayes(
    "CUSTOMER CHURN",
    churn_data,
    CHURN_FEATURES,
    ['Basic', 'Low', 'Many', 'Monthly', 'Yes'],
)


CUSTOMER CHURN
Input: {'Plan': 'Basic', 'Usage': 'Low', 'Tickets': 'Many', 'Contract': 'Monthly', 'AutoPay': 'Yes'}
Du doan: Yes
Do tin cay: 87.7%


In [6]:
df = pd.DataFrame(churn_data, columns=CHURN_FEATURES + ['Status'])

In [7]:
model_churn, pred_churn, probs_churn = run_naive_bayes(
    "CUSTOMER CHURN",
    df,
    CHURN_FEATURES,
    ['Basic', 'Low', 'Many', 'Monthly', 'Yes'],
)


CUSTOMER CHURN
Input: {'Plan': 'Basic', 'Usage': 'Low', 'Tickets': 'Many', 'Contract': 'Monthly', 'AutoPay': 'Yes'}
Du doan: Yes
Do tin cay: 87.7%


In [1]:
"""Chạy: python make_excel_templates.py  → tạo thư mục excel/ với 3 workbook."""
from pathlib import Path
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation

OUT = Path("excel"); OUT.mkdir(exist_ok=True)
HEAD = PatternFill("solid", fgColor="1F4E78"); HFONT = Font(bold=True, color="FFFFFF")
INPUT = PatternFill("solid", fgColor="FFF2CC")     # ô vàng = nhập
CALC = PatternFill("solid", fgColor="E2EFDA")      # ô xanh = công thức
BOLD = Font(bold=True)


def new_wb():
    wb = Workbook(); wb.remove(wb.active); return wb


def header(ws, row, labels):
    for j, lab in enumerate(labels, 1):
        c = ws.cell(row=row, column=j, value=lab)
        c.fill, c.font = HEAD, HFONT
        c.alignment = Alignment(horizontal="center", wrap_text=True)


def widths(ws, ws_widths):
    for i, w in enumerate(ws_widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = w


def put(ws, ref, value, fill=None, fmt=None, bold=False):
    c = ws[ref]; c.value = value
    if fill: c.fill = fill
    if fmt: c.number_format = fmt
    if bold: c.font = BOLD
    return c


# =====================================================================
# 1) BAYES
# =====================================================================
def bayes_sheet(wb, title, question, rows, n_rows=None):
    ws = wb.create_sheet(title)
    put(ws, "A1", question, bold=True)
    put(ws, "A2", "Ô vàng = nhập; ô xanh = công thức.  H = giả thuyết (nguyên nhân ẩn), E = bằng chứng.")
    header(ws, 4, ["H (giả thuyết)", "Prior P(H)", "Likelihood P(E|H)", "Prior × Likelihood", "Posterior P(H|E)"])
    n = n_rows or len(rows)
    first, last = 5, 4 + n
    tot = last + 1
    for i in range(n):
        r = first + i
        name, prior, lik = rows[i] if i < len(rows) else (None, None, None)
        for col, val in zip("ABC", (name, prior, lik)):
            ws[f"{col}{r}"].value = val; ws[f"{col}{r}"].fill = INPUT
        put(ws, f"D{r}", f'=IF(OR(B{r}="",C{r}=""),"",B{r}*C{r})', CALC, "0.00000")
        put(ws, f"E{r}", f'=IF(D{r}="","",D{r}/$D${tot})', CALC, "0.0000")
    put(ws, f"A{tot}", "Tổng", bold=True)
    put(ws, f"B{tot}", f"=SUM(B{first}:B{last})", CALC)
    put(ws, f"D{tot}", f"=SUM(D{first}:D{last})", CALC, "0.00000", True)
    put(ws, f"E{tot}", f"=SUM(E{first}:E{last})", CALC, "0.0000")
    put(ws, f"A{tot+2}", "P(E) = evidence = tổng cột D"); put(ws, f"B{tot+2}", f"=D{tot}", CALC, "0.00000")
    put(ws, f"A{tot+3}", "Kiểm tra Σ prior = 1?")
    put(ws, f"B{tot+3}", f'=IF(ABS(B{tot}-1)<0.000001,"OK","SAI: tổng prior khác 1")', CALC)
    put(ws, f"A{tot+4}", "Giả thuyết khả nghi nhất")
    put(ws, f"B{tot+4}", f"=INDEX(A{first}:A{last},MATCH(MAX(E{first}:E{last}),E{first}:E{last},0))", CALC, bold=True)
    widths(ws, [34, 14, 20, 20, 18])
    return ws


def make_bayes():
    wb = new_wb()
    bayes_sheet(wb, "Template", "TEMPLATE: điền H, prior, likelihood (tối đa 5 giả thuyết)", [], n_rows=5)
    bayes_sheet(wb, "VD1_YTe", "VD1: 1% mắc bệnh; test đúng 90%; dương tính giả 5%. Dương tính → P(bệnh)?",
                [("Bệnh", 0.01, 0.90), ("Khỏe", 0.99, 0.05)])
    bayes_sheet(wb, "VD2_Spam", "VD2: 40% email là spam; 'FREE' có trong 60% spam, 5% ham. Có 'FREE' → P(spam)?",
                [("Spam", 0.4, 0.60), ("Ham", 0.6, 0.05)])
    bayes_sheet(wb, "VD3_SanXuat", "VD3: Dây chuyền A/B/C = 50/30/20% sản lượng; lỗi 1/2/3%. Sản phẩm lỗi → từ đâu?",
                [("A", 0.5, 0.01), ("B", 0.3, 0.02), ("C", 0.2, 0.03)])
    bayes_sheet(wb, "VD4_GianLan", "VD4: 0.2% giao dịch gian lận; rule bắt 95%, báo nhầm 3%. Bị báo → P(gian lận)?",
                [("Gian lận", 0.002, 0.95), ("Bình thường", 0.998, 0.03)])
    bayes_sheet(wb, "VD5_Lead", "VD5: 20% lead mua; 70% người mua mở email, 25% người không mua mở. Mở → P(mua)?",
                [("Mua", 0.2, 0.70), ("Không mua", 0.8, 0.25)])
    bayes_sheet(wb, "BT_Giao_hang", "BT: Kho HN/ĐN/HCM = 20/30/50% đơn; trễ 12/3/4%. Đơn trễ → kho nào? (tự điền)",
                [("Hà Nội", None, None), ("Đà Nẵng", None, None), ("HCM", None, None)])

    ws = wb.create_sheet("CapNhatTuanTu")
    put(ws, "A1", "VD3 cập nhật tuần tự: posterior sau lỗi #1 trở thành prior cho lỗi #2", bold=True)
    header(ws, 4, ["H", "Prior", "Likelihood lỗi", "Posterior sau lỗi #1", "Posterior sau lỗi #2"])
    for i, (h, p, l) in enumerate([("A", .5, .01), ("B", .3, .02), ("C", .2, .03)]):
        r = 5 + i
        ws[f"A{r}"], ws[f"B{r}"], ws[f"C{r}"] = h, p, l
        ws[f"B{r}"].fill = ws[f"C{r}"].fill = INPUT
        put(ws, f"D{r}", f"=B{r}*C{r}/SUMPRODUCT($B$5:$B$7,$C$5:$C$7)", CALC, "0.0000")
        put(ws, f"E{r}", f"=D{r}*C{r}/SUMPRODUCT($D$5:$D$7,$C$5:$C$7)", CALC, "0.0000")
    widths(ws, [10, 12, 16, 22, 22])

    ws = wb.create_sheet("TanSuatTuNhien")
    put(ws, "A1", "Cách đếm bằng N người (giải thích VD1)", bold=True)
    for r, (lab, val) in enumerate([("Số người N", 10000), ("Prior (tỷ lệ bệnh)", 0.01),
                                     ("P(+|bệnh) độ nhạy", 0.90), ("P(+|khỏe) dương tính giả", 0.05)], 3):
        ws[f"A{r}"] = lab; put(ws, f"B{r}", val, INPUT)
    for r, (lab, f) in enumerate([("Số người bệnh", "=B3*B4"), ("→ dương tính thật", "=B8*B5"),
                                   ("Số người khỏe", "=B3-B8"), ("→ dương tính giả", "=B10*B6"),
                                   ("P(bệnh | +) = TP/(TP+FP)", "=B9/(B9+B11)")], 8):
        ws[f"A{r}"] = lab; put(ws, f"B{r}", f, CALC, "0.0000" if r == 12 else "#,##0")
    widths(ws, [34, 16])
    wb.save(OUT / "Bayes_template.xlsx")


# =====================================================================
# 2) A/B: Bài 1 + Bài 2
# =====================================================================
def make_ab():
    wb = new_wb()
    ws = wb.create_sheet("Bai1")
    put(ws, "A1", "BÀI 1 – Conversion rate: z-test 2 tỷ lệ (ô vàng = nhập)", bold=True)
    header(ws, 3, ["", "Control", "Variant"])
    ws["A4"], ws["A5"], ws["A6"] = "n (visitor)", "x (số đơn)", "CR = x/n"
    for ref, v in (("B4", 13000), ("C4", 13000), ("B5", 650), ("C5", 754)):
        put(ws, ref, v, INPUT)
    put(ws, "B6", "=B5/B4", CALC, "0.00%"); put(ws, "C6", "=C5/C4", CALC, "0.00%")
    items = [
        (8, "diff = CR_V − CR_C", "=C6-B6", "0.0000"),
        (9, "Lift tương đối", "=B8/B6", "0.0%"),
        (10, "p_pool", "=(B5+C5)/(B4+C4)", "0.0000"),
        (11, "SE (pooled)", "=SQRT(B10*(1-B10)*(1/B4+1/C4))", "0.000000"),
        (12, "z", "=B8/B11", "0.000"),
        (13, "p-value (2 phía)", "=2*(1-NORMSDIST(ABS(B12)))", "0.0000"),
        (14, "SE (unpooled, cho CI)", "=SQRT(B6*(1-B6)/B4+C6*(1-C6)/C4)", "0.000000"),
        (15, "CI 95% – cận dưới", "=B8-NORMSINV(0.975)*B14", "0.0000"),
        (16, "CI 95% – cận trên", "=B8+NORMSINV(0.975)*B14", "0.0000"),
    ]
    for r, lab, f, fmt in items:
        ws[f"A{r}"] = lab; put(ws, f"B{r}", f, CALC, fmt)
    ws["A18"] = "α"; put(ws, "B18", 0.05, INPUT)
    ws["A17"] = "Kết luận"
    put(ws, "B17", '=IF(B13<B18,IF(B8>0,"Variant THẮNG","Variant THUA"),"Chưa đủ bằng chứng")', CALC, bold=True)

    put(ws, "A20", "CỠ MẪU TỐI THIỂU MỖI NHÓM (ROUNDUP = ceil)", bold=True)
    items = [
        (21, "α", "=B18", None), (22, "Power", 0.8, "in"), (23, "p1 baseline", 0.05, "in"),
        (24, "MDE tuyệt đối", 0.008, "in"), (25, "p2 = p1 + MDE", "=B23+B24", None),
        (26, "p̄ = (p1+p2)/2", "=(B23+B25)/2", None),
        (27, "Z(α/2)", "=NORMSINV(1-B21/2)", None), (28, "Z(power)", "=NORMSINV(B22)", None),
        (29, "n MỖI NHÓM",
         "=ROUNDUP(((B27*SQRT(2*B26*(1-B26))+B28*SQRT(B23*(1-B23)+B25*(1-B25)))^2)/(B24^2),0)", None),
        (30, "Lưu lượng/ngày (cả 2 nhóm)", 2000, "in"),
        (31, "Số ngày tối thiểu", "=ROUNDUP(B29*2/B30,0)", None),
        (32, "Làm tròn lên bội 7 ngày", "=ROUNDUP(B31/7,0)*7", None),
    ]
    for r, lab, f, kind in items:
        ws[f"A{r}"] = lab
        put(ws, f"B{r}", f, INPUT if kind == "in" else CALC, "#,##0" if r in (29, 30, 31, 32) else "0.0000")
    ws["B29"].font = BOLD
    widths(ws, [32, 16, 16])

    ws = wb.create_sheet("Bai2")
    put(ws, "A1", "Control", bold=True); put(ws, "B1", "Variant", bold=True)
    ctrl = [20, 25, 30, 22, 28, 35, 24, 26]; var = [28, 32, 30, 27, 35, 40, 29, 31]
    for i, (a, b) in enumerate(zip(ctrl, var), 2):
        put(ws, f"A{i}", a, INPUT); put(ws, f"B{i}", b, INPUT)
    header(ws, 1, ["Control", "Variant", "", "Chỉ số", "Control", "Variant"])
    put(ws, "H1", "← BÀI 2: AOV, Welch t-test", bold=True)
    rows = [
        (2, "n", "=COUNT(A2:A9)", "=COUNT(B2:B9)"), (3, "Trung bình", "=AVERAGE(A2:A9)", "=AVERAGE(B2:B9)"),
        (4, "Phương sai", "=VAR(A2:A9)", "=VAR(B2:B9)"),
    ]
    for r, lab, fc, fv in rows:
        ws[f"D{r}"] = lab; put(ws, f"E{r}", fc, CALC, "0.00"); put(ws, f"F{r}", fv, CALC, "0.00")
    single = [
        (5, "diff = TB_V − TB_C", "=F3-E3"), (6, "SE (Welch)", "=SQRT(E4/E2+F4/F2)"), (7, "t", "=E5/E6"),
        (8, "df (Welch)", "=(E6^2)^2/((E4/E2)^2/(E2-1)+(F4/F2)^2/(F2-1))"),
        (9, "p-value = TTEST(...,2,3)", "=TTEST(A2:A9,B2:B9,2,3)"),
        (10, "t tới hạn (TINV cắt df xuống số nguyên)", "=TINV(0.05,E8)"),
        (11, "CI 95% – dưới", "=E5-E10*E6"), (12, "CI 95% – trên", "=E5+E10*E6"),
        (14, "σ² chung = (var_C + var_V)/2", "=(E4+F4)/2"), (15, "MDE ($)", 5),
        (16, "n mỗi nhóm cần có",
         "=ROUNDUP(2*(NORMSINV(0.975)+NORMSINV(0.8))^2*E14/E15^2,0)"),
    ]
    for r, lab, f in single:
        ws[f"D{r}"] = lab; put(ws, f"E{r}", f, INPUT if r == 15 else CALC, "0.0000")
    widths(ws, [10, 10, 3, 40, 12, 12])
    wb.save(OUT / "AB_Bai1_Bai2_template.xlsx")


# =====================================================================
# 3) TENNIS NAIVE BAYES
# =====================================================================
TENNIS = [
    ("Overcast", "Hot", "High", "Weak", "Yes"), ("Sunny", "Hot", "High", "Strong", "No"),
    ("Rain", "Cool", "Normal", "Weak", "Yes"), ("Sunny", "Mild", "High", "Strong", "No"),
    ("Overcast", "Cool", "Normal", "Strong", "Yes"), ("Rain", "Cool", "Normal", "Strong", "No"),
    ("Sunny", "Cool", "Normal", "Weak", "Yes"), ("Sunny", "Hot", "High", "Weak", "No"),
    ("Rain", "Mild", "High", "Weak", "Yes"), ("Sunny", "Mild", "Normal", "Strong", "Yes"),
]
FEATS = [("Outlook", "A", ["Sunny", "Overcast", "Rain"]), ("Temp", "B", ["Hot", "Mild", "Cool"]),
         ("Humidity", "C", ["High", "Normal"]), ("Wind", "D", ["Weak", "Strong"])]


def make_tennis():
    wb = new_wb()
    ws = wb.create_sheet("Data")
    header(ws, 1, ["Outlook", "Temp", "Humidity", "Wind", "Play"])
    for i, row in enumerate(TENNIS, 2):
        for j, v in enumerate(row, 1):
            ws.cell(i, j, v).fill = INPUT
    widths(ws, [12, 10, 12, 10, 8])

    ws = wb.create_sheet("Bang_dem")
    put(ws, "A1", "alpha (Laplace: 0 = không, 1 = có)", bold=True); put(ws, "B1", 0, INPUT)
    put(ws, "A2", "n(Yes)"); put(ws, "B2", '=COUNTIF(Data!$E$2:$E$11,"Yes")', CALC)
    put(ws, "A3", "n(No)");  put(ws, "B3", '=COUNTIF(Data!$E$2:$E$11,"No")', CALC)
    put(ws, "A4", "P(Yes) prior"); put(ws, "B4", "=B2/(B2+B3)", CALC, "0.000")
    put(ws, "A5", "P(No) prior");  put(ws, "B5", "=B3/(B2+B3)", CALC, "0.000")
    header(ws, 7, ["Feature", "Value", "n(Yes)", "n(No)", "K (số giá trị)", "P(v|Yes)", "P(v|No)"])
    r = 8
    for feat, col, vals in FEATS:
        for v in vals:
            ws.cell(r, 1, feat); ws.cell(r, 2, v)
            rng = f"Data!${col}$2:${col}$11"
            put(ws, f"C{r}", f'=COUNTIFS({rng},B{r},Data!$E$2:$E$11,"Yes")', CALC)
            put(ws, f"D{r}", f'=COUNTIFS({rng},B{r},Data!$E$2:$E$11,"No")', CALC)
            put(ws, f"E{r}", f"=COUNTIF($A$8:$A$17,A{r})", CALC)
            put(ws, f"F{r}", f"=(C{r}+$B$1)/($B$2+$B$1*E{r})", CALC, "0.0000")
            put(ws, f"G{r}", f"=(D{r}+$B$1)/($B$3+$B$1*E{r})", CALC, "0.0000")
            r += 1
    widths(ws, [34, 12, 9, 9, 14, 11, 11])

    ws = wb.create_sheet("Predict")
    header(ws, 1, ["Feature", "Giá trị (chọn)", "P(v|Yes)", "P(v|No)"])
    query = ["Sunny", "Cool", "High", "Strong"]
    for i, ((feat, _, vals), q) in enumerate(zip(FEATS, query), 2):
        ws[f"A{i}"] = feat; put(ws, f"B{i}", q, INPUT)
        dv = DataValidation(type="list", formula1='"' + ",".join(vals) + '"', allow_blank=False)
        ws.add_data_validation(dv); dv.add(f"B{i}")
        put(ws, f"C{i}", f"=SUMIFS(Bang_dem!$F$8:$F$17,Bang_dem!$A$8:$A$17,$A{i},Bang_dem!$B$8:$B$17,$B{i})", CALC, "0.0000")
        put(ws, f"D{i}", f"=SUMIFS(Bang_dem!$G$8:$G$17,Bang_dem!$A$8:$A$17,$A{i},Bang_dem!$B$8:$B$17,$B{i})", CALC, "0.0000")
    ws["A7"] = "Prior"; put(ws, "C7", "=Bang_dem!B4", CALC, "0.000"); put(ws, "D7", "=Bang_dem!B5", CALC, "0.000")
    ws["A8"] = "Điểm = prior × tích"
    put(ws, "C8", "=C7*PRODUCT(C2:C5)", CALC, "0.00000"); put(ws, "D8", "=D7*PRODUCT(D2:D5)", CALC, "0.00000")
    ws["A9"] = "Posterior (chuẩn hóa)"
    put(ws, "C9", '=IF(C8+D8=0,"n/a",C8/(C8+D8))', CALC, "0.0000")
    put(ws, "D9", '=IF(C8+D8=0,"n/a",D8/(C8+D8))', CALC, "0.0000")
    ws["A10"] = "Quyết định"
    put(ws, "B10", '=IF(C8+D8=0,"n/a",IF(D8>C8,"Should not go!","Go play!"))', CALC, bold=True)
    ws["A12"] = "Thử: (1) alpha=0 vs 1 ở Bang_dem!B1  (2) Outlook=Overcast để thấy zero-frequency."
    widths(ws, [26, 18, 12, 12])
    wb.save(OUT / "Tennis_NaiveBayes_template.xlsx")


if __name__ == "__main__":
    make_bayes(); make_ab(); make_tennis()
    print("Đã tạo:", *[p.name for p in OUT.glob("*.xlsx")])

Đã tạo: AB_Bai1_Bai2_template.xlsx Bayes_template.xlsx Tennis_NaiveBayes_template.xlsx


In [4]:
def make_ab():
    wb = new_wb()
    ws = wb.create_sheet("Bai1")
    put(ws, "A1", "BÀI 1 – Trang mới có làm nhiều người mua hơn không? (vàng = nhập, xanh = công thức)", bold=True)
    header(ws, 3, ["", "Nhóm A (cũ)", "Nhóm B (mới)"])
    ws["A4"], ws["A5"], ws["A6"] = "Số người vào xem", "Số người mua", "Tỷ lệ mua"
    for ref, v in (("B4", 13000), ("C4", 13000), ("B5", 650), ("C5", 754)):
        put(ws, ref, v, INPUT, "#,##0")
    put(ws, "B6", "=B5/B4", CALC, "0.00%"); put(ws, "C6", "=C5/C4", CALC, "0.00%")
    rows = [
        (8,  "Chênh lệch (B − A)",             "=C6-B6", "0.0000"),
        (9,  "Tăng bao nhiêu % so với A",       "=B8/B6", "0.0%"),
        (10, "Sai số (SE)",                     "=SQRT(B6*(1-B6)/B4+C6*(1-C6)/C4)", "0.000000"),
        (11, "z = chênh lệch / sai số",         "=B8/B10", "0.000"),
        (12, "p-value",                         "=2*(1-NORMSDIST(ABS(B11)))", "0.0000"),
        (13, "Khoảng tin cậy 95% – từ",         "=B8-1.96*B10", "0.0000"),
        (14, "Khoảng tin cậy 95% – đến",        "=B8+1.96*B10", "0.0000"),
    ]
    for r, lab, f, fmt in rows:
        ws[f"A{r}"] = lab; put(ws, f"B{r}", f, CALC, fmt)
    ws["A15"] = "Kết luận"
    put(ws, "B15", '=IF(B12<0.05,IF(B8>0,"B tốt hơn thật","B tệ hơn thật"),"Chưa đủ bằng chứng")', CALC, bold=True)

    put(ws, "A17", "CẦN BAO NHIÊU NGƯỜI MỖI NHÓM? (quy tắc 16, ước lượng nhanh)", bold=True)
    ws["A18"], ws["A19"] = "Tỷ lệ mua hiện tại", "Muốn phát hiện tăng tối thiểu (MDE)"
    put(ws, "B18", 0.05, INPUT, "0.0%"); put(ws, "B19", 0.008, INPUT, "0.0%")
    ws["A20"] = "Số người mỗi nhóm (làm tròn lên)"
    put(ws, "B20", "=ROUNDUP(16*(B18+B19/2)*(1-(B18+B19/2))/B19^2,0)", CALC, "#,##0", True)
    ws["A21"] = "Khách mỗi ngày (cả 2 nhóm)"; put(ws, "B21", 2000, INPUT, "#,##0")
    ws["A22"] = "Số ngày cần chạy";            put(ws, "B22", "=ROUNDUP(B20*2/B21,0)", CALC, "#,##0")
    ws["A24"] = "Ghi chú: Python (sample_size_rate) tính chính xác hơn, lệch khoảng 2%."
    widths(ws, [44, 16, 16])

    ws = wb.create_sheet("Bai2")
    header(ws, 1, ["Nhóm A", "Nhóm B", "", "Chỉ số", "Nhóm A", "Nhóm B"])
    ctrl = [20, 25, 30, 22, 28, 35, 24, 26]; var = [28, 32, 30, 27, 35, 40, 29, 31]
    for i, (a, b) in enumerate(zip(ctrl, var), 2):
        put(ws, f"A{i}", a, INPUT); put(ws, f"B{i}", b, INPUT)
    ws["D2"], ws["D3"], ws["D4"] = "Số đơn", "Trung bình", "Độ lệch chuẩn"
    for col, rng in (("E", "A2:A9"), ("F", "B2:B9")):
        put(ws, f"{col}2", f"=COUNT({rng})", CALC)
        put(ws, f"{col}3", f"=AVERAGE({rng})", CALC, "0.00")
        put(ws, f"{col}4", f"=STDEV({rng})", CALC, "0.00")
    ws["D6"] = "Chênh lệch (B − A)";            put(ws, "E6", "=F3-E3", CALC, "0.00")
    ws["D7"] = "p-value (TTEST loại 3)";        put(ws, "E7", "=TTEST(A2:A9,B2:B9,2,3)", CALC, "0.0000")
    ws["D8"] = "Kết luận"
    put(ws, "E8", '=IF(E7<0.05,IF(E6>0,"B cao hơn","B thấp hơn"),"Chưa đủ bằng chứng")', CALC, bold=True)
    ws["D10"] = "Muốn phát hiện chênh lệch tối thiểu ($)"; put(ws, "E10", 5, INPUT)
    ws["D11"] = "Độ lệch chuẩn dùng để tính";              put(ws, "E11", "=(E4+F4)/2", CALC, "0.00")
    ws["D12"] = "Số đơn mỗi nhóm cần có (quy tắc 16)"
    put(ws, "E12", "=ROUNDUP(16*E11^2/E10^2,0)", CALC, "#,##0", True)
    ws["D14"] = "Nhận xét: chỉ có 8 đơn mỗi nhóm → kết luận chỉ mang tính gợi ý."
    widths(ws, [10, 10, 3, 40, 12, 12])
    wb.save(OUT / "AB_Bai1_Bai2_template_v2.xlsx")

In [5]:
make_ab()